In [42]:
!pip install -q seaborn pandas matplotlib

ERROR: Could not find a version that satisfies the requirement seaborn (from versions: none)
ERROR: No matching distribution found for seaborn


# Cell 1: 环境配置、依赖导入与云盘路径设定

In [43]:
%pip install pandas openpyxl opencc-python-reimplemented -i https://pypi.tuna.tsinghua.edu.cn/simple

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn.functional as F
import math
import os
import json
import glob
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd  # 用于导出 Excel
import opencc
from tqdm import tqdm

 
# 云盘保存路径设置
 
# 云数据盘路径，默认在当前目录下新建 output 文件夹
CLOUD_DISK_PATH = "./poetry_output"  

if not os.path.exists(CLOUD_DISK_PATH):
    os.makedirs(CLOUD_DISK_PATH)
    print(f"📁 已自动创建云盘保存目录: {CLOUD_DISK_PATH}")

# 设置随机种子，检测 GPU
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"当前计算设备: {device} | 所有产出将自动保存至: {CLOUD_DISK_PATH}")

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simpleNote: you may need to restart the kernel to use updated packages.

🚀 当前计算设备: cuda | 所有产出将自动保存至: ./poetry_output



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: E:\anaconda3\python.exe -m pip install --upgrade pip


# Cell 2: 解析本地 JSON 数据集、构建词表与数据集划分

In [44]:
class JsonPoetryDataset(Dataset):
    def __init__(self, data_dir, max_len=64, min_freq=2):
        self.max_len = max_len
        self.poems = []
        self.word2id = {'<PAD>': 0, '<START>': 1, '<END>': 2, '<UNK>': 3}
        self.id2word = {0: '<PAD>', 1: '<START>', 2: '<END>', 3: '<UNK>'}
        word_counts = {}
        
        search_pattern = os.path.join(data_dir, "*.json")
        json_files = glob.glob(search_pattern)
        print(f"🔍 找到 {len(json_files)} 个 JSON 数据文件，正在解析...")
        
        for file_path in json_files:
            try:
                with open(file_path, "r", encoding="utf-8") as f:
                    data = json.load(f)
                    for poem_data in data:
                        paragraphs = poem_data.get('paragraphs', [])
                        content = "".join(paragraphs)
                        
                        if not content or any(char in content for char in ['_', '(', ')', '（', '）', '《', '》', '[', ']']):
                            continue
                        if 10 <= len(content) <= max_len - 2:
                            self.poems.append(content)
                            for word in content:
                                word_counts[word] = word_counts.get(word, 0) + 1
            except Exception as e:
                pass 
                
        for word, count in word_counts.items():
            if count >= min_freq:
                idx = len(self.word2id)
                self.word2id[word] = idx
                self.id2word[idx] = word
                
        self.vocab_size = len(self.word2id)

    def __len__(self): return len(self.poems)

    def __getitem__(self, idx):
        poem = self.poems[idx]
        poem_ids = [self.word2id['<START>']] + [self.word2id.get(w, self.word2id['<UNK>']) for w in poem] + [self.word2id['<END>']]
        pad_len = self.max_len - len(poem_ids)
        poem_ids = poem_ids + [self.word2id['<PAD>']] * pad_len
        return torch.tensor(poem_ids[:-1], dtype=torch.long), torch.tensor(poem_ids[1:], dtype=torch.long)

# 数据集 JSON 路径
data_directory = "./chinese-poetry-master/全唐诗" 
dataset = JsonPoetryDataset(data_dir=data_directory, max_len=64, min_freq=3)

if len(dataset) > 0:
    # 2. 8:1:1 数据集划分
    total_len = len(dataset)
    train_size = int(0.8 * total_len)
    val_size = int(0.1 * total_len)
    test_size = total_len - train_size - val_size
    
    train_dataset, val_dataset, test_dataset = random_split(
        dataset, [train_size, val_size, test_size]
    )
    
    # 3. 配置 DataLoader (注意 Batch Size 和 drop_last)
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, drop_last=False)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, drop_last=False)
    
    print(f"✅ 数据集划分完毕！")
    print(f"📊 训练集: {len(train_dataset)} | 验证集: {len(val_dataset)} | 测试集: {len(test_dataset)}")
else:
    print("❌ 警告：未读取到数据，请检查 data_directory 路径！")

🔍 找到 318 个 JSON 数据文件，正在解析...
✅ 数据集划分完毕！
📊 训练集: 149831 | 验证集: 18728 | 测试集: 18730


# Cell 3: 搭建深度 GPT 架构 Transformer 模型

In [45]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x): 
        return self.dropout(x + self.pe[:, :x.size(1)])

class GPTStylePoetryModel(nn.Module):
    def __init__(self, vocab_size, d_model=512, nhead=8, num_layers=6, dropout=0.2):
        super(GPTStylePoetryModel, self).__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout)
        
        encoder_layers = nn.TransformerEncoderLayer(
            d_model, nhead, dim_feedforward=d_model*4, 
            dropout=dropout, batch_first=True, activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layers, num_layers)
        self.ln_f = nn.LayerNorm(d_model)
        self.fc_out = nn.Linear(d_model, vocab_size, bias=False)
        self.fc_out.weight = self.embedding.weight

    def forward(self, src):
        seq_len = src.size(1)
        # 3. 标准 Transformer Mask (使用 -inf)
        mask = torch.triu(torch.ones(seq_len, seq_len, device=src.device), diagonal=1).bool()
        mask_inf = torch.zeros(seq_len, seq_len, device=src.device).masked_fill(mask, float('-inf'))
        
        x = self.pos_encoder(self.embedding(src) * math.sqrt(self.d_model))
        output = self.transformer(x, mask_inf)
        return self.fc_out(self.ln_f(output))

# model = GPTStylePoetryModel(vocab_size=dataset.vocab_size).to(device)
model = GPTStylePoetryModel(
    vocab_size=dataset.vocab_size,
).to(device)
print(f"✅ 模型初始化完成！总参数量: {sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6:.2f} M")

✅ 模型初始化完成！总参数量: 22.99 M


# Cell 4: 训练循环、数据保存

In [46]:
epochs = 30  
criterion = nn.CrossEntropyLoss(ignore_index=dataset.word2id['<PAD>'])
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)

scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=3e-4,
    steps_per_epoch=len(train_loader),
    epochs=epochs,
    pct_start=0.1,
    anneal_strategy='cos',
    div_factor=10,
    final_div_factor=100
)

# 记录器 (额外记录 Epoch 用于导出 Excel)
history = {'Epoch': [], 'Train_Loss': [], 'Val_Loss': [], 'Train_PPL': [], 'Val_PPL': []}
best_val_loss = float('inf')

print(f"🔥 开始训练！所有模型和数据将实时保存至: {CLOUD_DISK_PATH}")

for epoch in range(epochs):
    # --- 训练阶段 ---
    model.train()
    total_train_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", unit="batch", ncols=100)
    
    for x, y in pbar:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        
        outputs = model(x)
        loss = criterion(outputs.view(-1, dataset.vocab_size), y.view(-1))
        
        loss.backward()
        # 2. 梯度裁剪防止梯度爆炸
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        
        optimizer.step()
        # 3. 每个 batch 调用一次 scheduler
        scheduler.step()
        
        total_train_loss += loss.item()
        pbar.set_postfix({"loss": f"{loss.item():.4f}", "lr": f"{scheduler.get_last_lr()[0]:.6f}"})
            
    avg_train_loss = total_train_loss / len(train_loader)
    
    # --- 验证阶段 ---
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for x_val, y_val in val_loader:
            x_val, y_val = x_val.to(device), y_val.to(device)
            loss = criterion(model(x_val).view(-1, dataset.vocab_size), y_val.view(-1))
            total_val_loss += loss.item()
            
    avg_val_loss = total_val_loss / max(1, len(val_loader))
    
    # 计算困惑度 PPL
    train_ppl = math.exp(avg_train_loss) if avg_train_loss < 20 else float('inf')
    val_ppl = math.exp(avg_val_loss) if avg_val_loss < 20 else float('inf')
    
    # 记录本轮数据
    history['Epoch'].append(epoch + 1)
    history['Train_Loss'].append(round(avg_train_loss, 4))
    history['Val_Loss'].append(round(avg_val_loss, 4))
    history['Train_PPL'].append(round(train_ppl, 2))
    history['Val_PPL'].append(round(val_ppl, 2))
    
    print(f"✅ Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val PPL: {val_ppl:.2f}")

    # 1. 保存最优模型
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_model_path = os.path.join(CLOUD_DISK_PATH, "poetry_gpt_best.pth")
        torch.save(model.state_dict(), best_model_path)
        print(f"   🌟 已保存最优模型至: {best_model_path}")
        
    # 2. 保存历史记录到 JSON
    with open(os.path.join(CLOUD_DISK_PATH, 'training_history.json'), 'w') as f:
        json.dump(history, f)

# =========================
# 训练结束后做最终测试评估
# =========================
best_model_path = os.path.join(CLOUD_DISK_PATH, "poetry_gpt_best.pth")
model.load_state_dict(torch.load(best_model_path))
model.eval()

total_test_loss = 0

with torch.no_grad():
    for x_test, y_test in test_loader:
        x_test, y_test = x_test.to(device), y_test.to(device)
        outputs = model(x_test)
        loss = criterion(outputs.view(-1, dataset.vocab_size), y_test.view(-1))
        total_test_loss += loss.item()

avg_test_loss = total_test_loss / len(test_loader)
test_ppl = math.exp(avg_test_loss) if avg_test_loss < 20 else float('inf')

print(f"🎯 最终测试评估：Test Loss: {avg_test_loss:.4f} | Test PPL: {test_ppl:.2f}")

 
# 训练结束：自动出图与导出 Excel
 
print("\n🎉 训练完毕！正在整合所有绘图原始数据至 Excel...")

# 1. 准备训练指标数据
df_metrics = pd.DataFrame(history)

# 2. 使用 ExcelWriter 保存多表单 (Sheet)
excel_report_path = os.path.join(CLOUD_DISK_PATH, "experiment_full_data.xlsx")
with pd.ExcelWriter(excel_report_path, engine='openpyxl') as writer:
    # 保存训练过程数据 (用于还原 Loss 和 PPL 图)
    df_metrics.to_excel(writer, sheet_name='Training_Metrics', index=False)
    
print(f"📊 训练指标数据已存入 Sheet: [Training_Metrics] -> {excel_report_path}")

# 3. 导出为 Excel 和 CSV
df = pd.DataFrame(history)
excel_path = os.path.join(CLOUD_DISK_PATH, "training_data_report.xlsx")
csv_path = os.path.join(CLOUD_DISK_PATH, "training_data_report.csv")
df.to_excel(excel_path, index=False)
df.to_csv(csv_path, index=False, encoding='utf-8-sig') # 备用 csv 格式
print(f"📊 训练数据表格已导出至:\n  - {excel_path}\n  - {csv_path}")

epochs_list = history['Epoch']
plt.style.use('seaborn-v0_8-whitegrid')

# 4. 绘制并保存 Loss 曲线
plt.figure(figsize=(8, 5), dpi=150)
plt.plot(epochs_list, history['Train_Loss'], marker='o', label='Train Loss')
plt.plot(epochs_list, history['Val_Loss'], marker='s', label='Validation Loss')
plt.title('Model Training and Validation Loss', fontsize=14)
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Cross Entropy Loss', fontsize=12)
plt.legend()
plt.tight_layout()
loss_img_path = os.path.join(CLOUD_DISK_PATH, 'report_loss_curve.png')
plt.savefig(loss_img_path, dpi=300)
plt.close()

# 5. 绘制并保存 PPL 曲线
plt.figure(figsize=(8, 5), dpi=150)
plt.plot(epochs_list, history['Train_PPL'], marker='^', color='green', label='Train PPL')
plt.plot(epochs_list, history['Val_PPL'], marker='v', color='red', label='Validation PPL')
plt.title('Model Perplexity (PPL)', fontsize=14)
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Perplexity', fontsize=12)
plt.legend()
plt.tight_layout()
plt.yscale('log')
ppl_img_path = os.path.join(CLOUD_DISK_PATH, 'report_ppl_curve.png')
plt.savefig(ppl_img_path, dpi=300)
plt.close()

print("\n🔒 正在执行数据落盘操作 (sync)...")
os.system("sync") # 强制将内存缓存区的数据写入磁盘，防止关机丢数据

import json
import os
CLOUD_DISK_PATH = "./poetry_output"

# 将汉字与数字互相映射的字典存为 json
with open(os.path.join(CLOUD_DISK_PATH, 'word2id.json'), 'w', encoding='utf-8') as f:
    json.dump(dataset.word2id, f, ensure_ascii=False)
print("✅ 词表已成功导出为 word2id.json！")

print("✅ 所有报告素材已安全保存至云硬盘！即将关机。")
# os.system("shutdown -h now")                                               #关机###############################################################